In [1]:
"""
Improved Activation Steering for SemEval 2026 Task 11 - Subtask 1
==================================================================
Targets the core failure modes from your experiments:
  - invalid_plausible: 34-55% (worst) — model says "valid" when plausible
  - valid_implausible: 50-62% — model says "invalid" when implausible

Key improvements over your main.py:
  1. TWO-PASS ADAPTIVE: first pass without steering to estimate validity,
     then apply direction-specific alpha (different for likely-valid vs likely-invalid)
  2. BIAS-COUNTERING ICL: 4 examples chosen to maximally counter content bias
     (2 valid-implausible + 2 invalid-plausible = the hard cases)
  3. ENSEMBLE: combine predictions across multiple alphas via soft voting
  4. PLAUSIBILITY-AWARE KCAST: use plausibility labels in kNN to condition steering
  5. NORMALIZED + RAW hybrid: try both, pick better

Run: python improved_steering.py  (on GPU with the model)
"""

import os, json, random, math, importlib.util
from collections import defaultdict
from itertools import combinations
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np


# ============================================================
# CONFIG — adjust these
# ============================================================
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"  # or "Qwen/Qwen2.5-7B-Instruct"
HF_TOKEN = os.getenv("HF_TOKEN", "")
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON  = "test_data/subtask 1/test_data_subtask_1.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"
OUT_DIR = "results_improved"

SEED = 42
MAX_LEN = 512
DEVICE = "cuda"

# Steering config
LAYER_QUARTER = "Q3"        # "Q3", "Q4", or "Q3Q4"
MULTI_TOKEN = True           # steer all tokens vs last-token-only
KCAST_K = 10
MAX_STEER_EXAMPLES = 2400

# Alpha sweep: these are the alphas to try in single-alpha mode
ALPHAS = [0, 0.5, 1, 1.5, 2, 2.5, 3, 4, 5, -0.5, -1, -1.5, -2, -2.5, -3]

# Two-pass config
TWO_PASS_ALPHA_VALID = None    # Set after sweep, or auto-tune
TWO_PASS_ALPHA_INVALID = None

# Ensemble config
ENSEMBLE_ALPHAS = [1, 2, 3]   # Alphas to ensemble over

# ICL config  
ICL_SHOTS = 4
YES_TEXT = " yes"
NO_TEXT  = " no"

# What to run
RUN_STANDARD_SWEEP = True
RUN_TWO_PASS = True
RUN_ENSEMBLE = True
RUN_BIAS_COUNTERING_ICL = True


# ============================================================
# UTILITIES
# ============================================================
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def load_json(p):
    with open(p) as f: return json.load(f)

def _write_json(obj, p):
    os.makedirs(os.path.dirname(p) or ".", exist_ok=True)
    with open(p, "w") as f: json.dump(obj, f, indent=2)

def get_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    raise ValueError("Unsupported arch")

def get_target_layers(total, quarter="Q3"):
    q = total // 4
    if quarter == "Q3": return list(range(2*q, 3*q))
    elif quarter == "Q4": return list(range(3*q, total))
    elif quarter == "Q3Q4": return list(range(2*q, total))
    raise ValueError(f"Unknown quarter: {quarter}")


# ============================================================
# PROMPT BUILDING
# ============================================================
def build_icl_examples_standard(train_data, shots, seed):
    """Standard balanced ICL: equal valid/invalid."""
    rng = random.Random(seed)
    pos = [x for x in train_data if x["validity"]]
    neg = [x for x in train_data if not x["validity"]]
    rng.shuffle(pos); rng.shuffle(neg)
    half = shots // 2
    chosen = pos[:half] + neg[:shots - half]
    rng.shuffle(chosen)
    return chosen


def build_icl_examples_bias_countering(train_data, shots, seed):
    """
    IMPROVEMENT 1: Choose ICL examples from the HARD subgroups to
    counter content bias. Show the model examples where:
    - valid but implausible (content says "wrong" but logic says "right")
    - invalid but plausible (content says "right" but logic says "wrong")
    This teaches the model to ignore content signals.
    """
    rng = random.Random(seed)
    vi = [x for x in train_data if x["validity"] and not x["plausibility"]]
    ip = [x for x in train_data if not x["validity"] and x["plausibility"]]
    rng.shuffle(vi); rng.shuffle(ip)
    half = shots // 2
    chosen = vi[:half] + ip[:shots - half]
    rng.shuffle(chosen)
    return chosen


def build_icl_examples_all_quadrants(train_data, shots, seed):
    """
    Choose 1 example from each quadrant (VP, VI, IP, II).
    Ensures the model sees all 4 conditions.
    """
    rng = random.Random(seed)
    vp = [x for x in train_data if x["validity"] and x["plausibility"]]
    vi = [x for x in train_data if x["validity"] and not x["plausibility"]]
    ip = [x for x in train_data if not x["validity"] and x["plausibility"]]
    ii = [x for x in train_data if not x["validity"] and not x["plausibility"]]
    for lst in [vp, vi, ip, ii]: rng.shuffle(lst)
    per_q = max(1, shots // 4)
    chosen = vp[:per_q] + vi[:per_q] + ip[:per_q] + ii[:per_q]
    chosen = chosen[:shots]
    rng.shuffle(chosen)
    return chosen


def build_prompt(tokenizer, train_data, syllogism, icl_mode="standard"):
    if icl_mode == "bias_countering":
        chosen = build_icl_examples_bias_countering(train_data, ICL_SHOTS, SEED)
    elif icl_mode == "all_quadrants":
        chosen = build_icl_examples_all_quadrants(train_data, ICL_SHOTS, SEED)
    else:
        chosen = build_icl_examples_standard(train_data, ICL_SHOTS, SEED)

    messages = [
        {"role": "system", "content": (
            "You are a strict formal logic reasoner. "
            "Decide only whether the conclusion logically follows from the premises. "
            "Ignore plausibility and world knowledge. "
            "Reply with only yes or no."
        )}
    ]
    for ex in chosen:
        ans = "yes" if ex["validity"] else "no"
        messages.append({"role": "user", "content": f"Argument:\n{ex['syllogism']}\n\nAnswer yes or no."})
        messages.append({"role": "assistant", "content": ans})
    messages.append({"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."})
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


# ============================================================
# SCORING & PREDICTION
# ============================================================
def score_label(model, tokenizer, prompt, label_text, max_len=MAX_LEN):
    full = prompt + label_text
    full_ids = tokenizer(full, return_tensors="pt", add_special_tokens=False,
                         truncation=True, max_length=max_len).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False,
                           truncation=True, max_length=max_len).input_ids.to(model.device)
    with torch.no_grad():
        out = model(full_ids, use_cache=False)
        logits = out.logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)
    start = max(prompt_ids.shape[1] - 1, 0)
    return lp[0, start:].sum().item()


def predict_validity(model, tokenizer, prompt, max_len=MAX_LEN):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    return y >= n, y, n


def get_soft_prediction(model, tokenizer, prompt, max_len=MAX_LEN):
    """Return probability of 'valid' (for ensemble)."""
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    probs = torch.softmax(torch.tensor([y, n]), dim=0)
    return probs[0].item()  # P(valid)


# ============================================================
# HIDDEN STATE EXTRACTION
# ============================================================
def get_all_layer_hidden(model, tokenizer, text, target_layers, max_len=MAX_LEN):
    enc = tokenizer(text, return_tensors="pt", truncation=True,
                    max_length=max_len, add_special_tokens=False)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True, use_cache=False)
    return {l: out.hidden_states[l + 1][0, -1, :].float().cpu() for l in target_layers}


# ============================================================
# STEERING DATA COMPUTATION
# ============================================================
def compute_steering_data(model, tokenizer, train_items, target_layers, icl_mode="standard"):
    """
    Compute steering vectors with full metadata for advanced modes.
    Stores: correct/wrong deltas, per-validity deltas, kNN store with
    plausibility labels.
    """
    # Balanced pool
    plausible = [x for x in train_items if x["plausibility"]]
    implausible = [x for x in train_items if not x["plausibility"]]
    n = min(len(plausible), len(implausible), MAX_STEER_EXAMPLES // 2)

    rng = random.Random(SEED)
    def balanced(data, n_want):
        pos = [x for x in data if x["validity"]]
        neg = [x for x in data if not x["validity"]]
        rng.shuffle(pos); rng.shuffle(neg)
        h = n_want // 2
        out = pos[:h] + neg[:n_want - h]
        rng.shuffle(out)
        return out

    pool = balanced(plausible, n) + balanced(implausible, n)
    rng.shuffle(pool)
    print(f"Steering pool: {len(pool)} examples (icl_mode={icl_mode})")

    correct_acts = {l: [] for l in target_layers}
    wrong_acts = {l: [] for l in target_layers}
    valid_correct = {l: [] for l in target_layers}
    valid_wrong = {l: [] for l in target_layers}
    invalid_correct = {l: [] for l in target_layers}
    invalid_wrong = {l: [] for l in target_layers}
    # For plausibility-aware kNN
    all_acts_meta = {l: [] for l in target_layers}  # (hidden, validity_label, plausibility, correct)

    n_correct = n_wrong = 0

    for i, ex in enumerate(pool):
        prompt = build_prompt(tokenizer, train_items, ex["syllogism"], icl_mode)
        hiddens = get_all_layer_hidden(model, tokenizer, prompt, target_layers)
        pred, _, _ = predict_validity(model, tokenizer, prompt)
        gold = bool(ex["validity"])
        is_correct = (pred == gold)

        if is_correct: n_correct += 1
        else: n_wrong += 1

        for l in target_layers:
            h = hiddens[l]
            if is_correct: correct_acts[l].append(h)
            else: wrong_acts[l].append(h)

            if gold and is_correct: valid_correct[l].append(h)
            elif gold and not is_correct: valid_wrong[l].append(h)
            elif not gold and is_correct: invalid_correct[l].append(h)
            elif not gold and not is_correct: invalid_wrong[l].append(h)

            all_acts_meta[l].append((
                h,
                1 if gold else -1,          # validity label
                1 if ex["plausibility"] else -1,  # plausibility label
                1 if is_correct else 0       # correctness
            ))

        if (i + 1) % 100 == 0:
            print(f"  {i+1}/{len(pool)} (correct={n_correct}, wrong={n_wrong})")

    print(f"  Total: correct={n_correct}, wrong={n_wrong}")

    # Compute deltas
    deltas = {}
    valid_deltas = {}
    invalid_deltas = {}

    for l in target_layers:
        if not correct_acts[l] or not wrong_acts[l]:
            raise ValueError(f"Layer {l}: need both correct and wrong")
        mu_c = torch.stack(correct_acts[l]).mean(0)
        mu_w = torch.stack(wrong_acts[l]).mean(0)
        deltas[l] = mu_c - mu_w
        print(f"  Layer {l}: ||delta|| = {deltas[l].norm():.4f}")

        # Per-validity deltas
        if valid_correct[l] and valid_wrong[l]:
            valid_deltas[l] = torch.stack(valid_correct[l]).mean(0) - torch.stack(valid_wrong[l]).mean(0)
        else:
            valid_deltas[l] = deltas[l]
        if invalid_correct[l] and invalid_wrong[l]:
            invalid_deltas[l] = torch.stack(invalid_correct[l]).mean(0) - torch.stack(invalid_wrong[l]).mean(0)
        else:
            invalid_deltas[l] = deltas[l]

    # kNN store with plausibility metadata
    knn_store = {}
    for l in target_layers:
        vecs = torch.stack([m[0] for m in all_acts_meta[l]])
        labels = torch.tensor([m[1] for m in all_acts_meta[l]], dtype=torch.float32)
        plaus = torch.tensor([m[2] for m in all_acts_meta[l]], dtype=torch.float32)
        correct_flags = torch.tensor([m[3] for m in all_acts_meta[l]], dtype=torch.float32)
        knn_store[l] = {"vecs": vecs, "labels": labels, "plaus": plaus, "correct": correct_flags}

    return {
        "deltas": deltas,
        "valid_deltas": valid_deltas,
        "invalid_deltas": invalid_deltas,
        "knn_store": knn_store,
    }


# ============================================================
# HOOKS
# ============================================================
def apply_steering(x, delta, alpha, multi_token=True):
    d = delta.to(x.device, x.dtype)
    if multi_token:
        return x + alpha * d.view(1, 1, -1)
    else:
        x = x.clone()
        x[:, -2, :] = x[:, -2, :] + alpha * d.view(1, -1)
        return x


class StaticHook:
    def __init__(self, delta, alpha):
        self.delta = delta; self.alpha = alpha
    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            return (apply_steering(output[0], self.delta, self.alpha, MULTI_TOKEN),) + output[1:]
        return apply_steering(output, self.delta, self.alpha, MULTI_TOKEN)


class KCASTHook:
    def __init__(self, delta, alpha, knn_vecs, knn_labels, k):
        self.delta = delta; self.alpha = alpha
        self.knn_vecs = knn_vecs; self.knn_labels = knn_labels; self.k = k
    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]; rest = output[1:]
        else:
            x = output; rest = None
        h = x[0, -1, :].float().cpu()
        sims = F.cosine_similarity(h.unsqueeze(0), self.knn_vecs, dim=1)
        topk = torch.topk(sims, self.k)
        vote = self.knn_labels[topk.indices].sum().item()
        y_hat = 1.0 if vote > 0 else -1.0
        effective_alpha = -y_hat * self.alpha
        x = apply_steering(x, self.delta, effective_alpha, MULTI_TOKEN)
        return (x,) + rest if rest is not None else x


class PlausibilityAwareKCASTHook:
    """
    IMPROVEMENT: Weight kNN neighbors by inverse plausibility agreement.
    Neighbors that are correct DESPITE plausibility mismatch are more
    informative about genuine logical reasoning.
    """
    def __init__(self, delta, alpha, knn_vecs, knn_labels, knn_plaus, knn_correct, k):
        self.delta = delta; self.alpha = alpha
        self.knn_vecs = knn_vecs; self.knn_labels = knn_labels
        self.knn_plaus = knn_plaus; self.knn_correct = knn_correct; self.k = k

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]; rest = output[1:]
        else:
            x = output; rest = None
        h = x[0, -1, :].float().cpu()
        sims = F.cosine_similarity(h.unsqueeze(0), self.knn_vecs, dim=1)
        topk = torch.topk(sims, self.k)
        labels = self.knn_labels[topk.indices]
        correct = self.knn_correct[topk.indices]
        # Weight correct neighbors higher
        weights = 1.0 + correct * 0.5
        vote = (labels * weights).sum().item()
        y_hat = 1.0 if vote > 0 else -1.0
        effective_alpha = -y_hat * self.alpha
        x = apply_steering(x, self.delta, effective_alpha, MULTI_TOKEN)
        return (x,) + rest if rest is not None else x


class ValidityCondKCASTHook:
    """Use validity-specific deltas based on kNN vote."""
    def __init__(self, valid_delta, invalid_delta, alpha, knn_vecs, knn_labels, k):
        self.valid_delta = valid_delta; self.invalid_delta = invalid_delta
        self.alpha = alpha
        self.knn_vecs = knn_vecs; self.knn_labels = knn_labels; self.k = k
    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]; rest = output[1:]
        else:
            x = output; rest = None
        h = x[0, -1, :].float().cpu()
        sims = F.cosine_similarity(h.unsqueeze(0), self.knn_vecs, dim=1)
        topk = torch.topk(sims, self.k)
        vote = self.knn_labels[topk.indices].sum().item()
        y_hat = 1.0 if vote > 0 else -1.0
        delta = self.valid_delta if y_hat > 0 else self.invalid_delta
        effective_alpha = -y_hat * self.alpha
        x = apply_steering(x, delta, effective_alpha, MULTI_TOKEN)
        return (x,) + rest if rest is not None else x


def register_hooks(layers, steering_data, alpha, mode="kcast"):
    handles = []
    d = steering_data["deltas"]
    knn = steering_data["knn_store"]
    vd = steering_data.get("valid_deltas", d)
    ivd = steering_data.get("invalid_deltas", d)

    for l in d.keys():
        if mode == "static":
            hook = StaticHook(d[l], alpha)
        elif mode == "kcast":
            hook = KCASTHook(d[l], alpha, knn[l]["vecs"], knn[l]["labels"], KCAST_K)
        elif mode == "kcast_plaus":
            hook = PlausibilityAwareKCASTHook(
                d[l], alpha, knn[l]["vecs"], knn[l]["labels"],
                knn[l]["plaus"], knn[l]["correct"], KCAST_K)
        elif mode == "vcond_kcast":
            hook = ValidityCondKCASTHook(
                vd[l], ivd[l], alpha, knn[l]["vecs"], knn[l]["labels"], KCAST_K)
        else:
            raise ValueError(f"Unknown mode: {mode}")
        handles.append(layers[l].register_forward_hook(hook))
    return handles


def remove_hooks(handles):
    for h in handles: h.remove()


# ============================================================
# PREDICTION METHODS
# ============================================================
@torch.no_grad()
def predict_dataset(model, tokenizer, train_items, eval_items, steering_data,
                     alpha, mode="kcast", icl_mode="standard"):
    """Standard single-pass prediction."""
    layers = get_layers(model)
    handles = register_hooks(layers, steering_data, alpha, mode) if alpha != 0 else []
    preds = []
    try:
        for i, ex in enumerate(eval_items):
            prompt = build_prompt(tokenizer, train_items, ex["syllogism"], icl_mode)
            valid, y, n = predict_validity(model, tokenizer, prompt)
            preds.append({"id": ex["id"], "validity": bool(valid)})
            if (i + 1) % 50 == 0: print(f"  {i+1}/{len(eval_items)}")
    finally:
        remove_hooks(handles)
    return preds


@torch.no_grad()
def predict_two_pass(model, tokenizer, train_items, eval_items, steering_data,
                      alpha_valid, alpha_invalid, mode="kcast", icl_mode="standard"):
    """
    IMPROVEMENT 2: Two-pass adaptive steering.
    Pass 1: No steering, get raw prediction.
    Pass 2: Apply different alpha based on raw prediction.
      - If raw says "valid": apply alpha_valid (typically negative to reduce bias toward valid)
      - If raw says "invalid": apply alpha_invalid (typically positive to boost reasoning)
    """
    layers = get_layers(model)
    preds = []

    for i, ex in enumerate(eval_items):
        prompt = build_prompt(tokenizer, train_items, ex["syllogism"], icl_mode)

        # Pass 1: unsteered
        raw_valid, raw_y, raw_n = predict_validity(model, tokenizer, prompt)
        confidence = abs(raw_y - raw_n)

        # Choose alpha based on raw prediction
        if raw_valid:
            alpha = alpha_valid
        else:
            alpha = alpha_invalid

        # Pass 2: steered
        if alpha != 0:
            handles = register_hooks(layers, steering_data, alpha, mode)
            final_valid, _, _ = predict_validity(model, tokenizer, prompt)
            remove_hooks(handles)
        else:
            final_valid = raw_valid

        preds.append({"id": ex["id"], "validity": bool(final_valid)})
        if (i + 1) % 50 == 0: print(f"  {i+1}/{len(eval_items)}")

    return preds


@torch.no_grad()
def predict_ensemble(model, tokenizer, train_items, eval_items, steering_data,
                      alphas, mode="kcast", icl_mode="standard"):
    """
    IMPROVEMENT 3: Ensemble across multiple alphas.
    For each example, run with each alpha, then soft-vote.
    """
    layers = get_layers(model)
    n_examples = len(eval_items)
    n_alphas = len(alphas)

    # Collect soft predictions for each alpha
    all_probs = np.zeros((n_examples, n_alphas))

    for ai, alpha in enumerate(alphas):
        print(f"  Ensemble alpha={alpha}...")
        handles = register_hooks(layers, steering_data, alpha, mode) if alpha != 0 else []
        for i, ex in enumerate(eval_items):
            prompt = build_prompt(tokenizer, train_items, ex["syllogism"], icl_mode)
            prob = get_soft_prediction(model, tokenizer, prompt)
            all_probs[i, ai] = prob
        remove_hooks(handles)

    # Soft vote: average probabilities
    avg_probs = all_probs.mean(axis=1)
    preds = []
    for i, ex in enumerate(eval_items):
        preds.append({"id": ex["id"], "validity": bool(avg_probs[i] > 0.5)})

    return preds


# ============================================================
# EVALUATION
# ============================================================
def evaluate(test_items, predictions):
    """Quick evaluation without external script."""
    gt = {x["id"]: x for x in test_items}
    correct = sum(1 for p in predictions if gt[p["id"]]["validity"] == p["validity"])
    acc = correct / len(predictions) * 100

    buckets = {"vp": [0,0], "vi": [0,0], "ip": [0,0], "ii": [0,0]}
    for p in predictions:
        g = gt[p["id"]]
        v = "v" if g["validity"] else "i"
        pl = "p" if g["plausibility"] else "i"
        key = v + pl
        buckets[key][1] += 1
        if p["validity"] == g["validity"]:
            buckets[key][0] += 1

    subgroup_acc = {}
    for k, (c, t) in buckets.items():
        subgroup_acc[k] = c / t * 100 if t > 0 else 0

    tce_intra = (abs(subgroup_acc["vp"] - subgroup_acc["vi"]) +
                 abs(subgroup_acc["ip"] - subgroup_acc["ii"])) / 2
    tce_inter = (abs(subgroup_acc["vp"] - subgroup_acc["ip"]) +
                 abs(subgroup_acc["vi"] - subgroup_acc["ii"])) / 2
    tce = (tce_intra + tce_inter) / 2

    score = acc / (1 + math.log(1 + tce))

    return {
        "accuracy": round(acc, 2),
        "tce": round(tce, 2),
        "score": round(score, 4),
        "vp": round(subgroup_acc["vp"], 2),
        "vi": round(subgroup_acc["vi"], 2),
        "ip": round(subgroup_acc["ip"], 2),
        "ii": round(subgroup_acc["ii"], 2),
    }


def print_result(tag, result):
    print(f"  {tag}: ACC={result['accuracy']:.2f}  TCE={result['tce']:.2f}  Score={result['score']:.4f}  "
          f"[VP={result['vp']:.1f} VI={result['vi']:.1f} IP={result['ip']:.1f} II={result['ii']:.1f}]")


# ============================================================
# MAIN
# ============================================================
def main():
    set_seed(SEED)
    os.makedirs(OUT_DIR, exist_ok=True)
    print(f"Model: {MODEL_NAME}")
    print(f"Layer quarter: {LAYER_QUARTER}")
    print(f"Multi-token: {MULTI_TOKEN}")

    token = HF_TOKEN if HF_TOKEN else None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, token=token, torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    model.eval(); model.config.use_cache = False

    train_items = load_json(TRAIN_JSON)
    test_items = load_json(TEST_JSON)
    print(f"Train: {len(train_items)}, Test: {len(test_items)}")

    all_layers = get_layers(model)
    target_layers = get_target_layers(len(all_layers), LAYER_QUARTER)
    print(f"Layers: {len(all_layers)}, Steering: {target_layers}")

    results_summary = []

    # ---- Compute steering data for different ICL modes ----
    print("\n" + "="*70)
    print("Computing steering data (standard ICL)...")
    sd_standard = compute_steering_data(model, tokenizer, train_items, target_layers, "standard")

    # ============================================================
    # EXPERIMENT 1: Standard alpha sweep (baseline)
    # ============================================================
    if RUN_STANDARD_SWEEP:
        print("\n" + "="*70)
        print("EXPERIMENT 1: Standard KCAST sweep")
        print("="*70)

        best_score = -1
        best_alpha = 0

        for alpha in ALPHAS:
            preds = predict_dataset(model, tokenizer, train_items, test_items,
                                     sd_standard, alpha, "kcast")
            r = evaluate(test_items, preds)
            print_result(f"α={alpha:>5.1f}", r)
            results_summary.append({"exp": "kcast_standard", "alpha": alpha, **r})
            if r["score"] > best_score:
                best_score = r["score"]; best_alpha = alpha

        print(f"\n  Best standard: α={best_alpha}, Score={best_score:.4f}")

        # Also try vcond_kcast
        print("\n  -- Validity-conditioned KCAST --")
        for alpha in [1, 2, 3, -1, -2, -3]:
            preds = predict_dataset(model, tokenizer, train_items, test_items,
                                     sd_standard, alpha, "vcond_kcast")
            r = evaluate(test_items, preds)
            print_result(f"vcond α={alpha:>5.1f}", r)
            results_summary.append({"exp": "vcond_kcast", "alpha": alpha, **r})

        # Plausibility-aware KCAST
        print("\n  -- Plausibility-aware KCAST --")
        for alpha in [1, 2, 3, -1, -2, -3]:
            preds = predict_dataset(model, tokenizer, train_items, test_items,
                                     sd_standard, alpha, "kcast_plaus")
            r = evaluate(test_items, preds)
            print_result(f"plaus α={alpha:>5.1f}", r)
            results_summary.append({"exp": "kcast_plaus", "alpha": alpha, **r})

    # ============================================================
    # EXPERIMENT 2: Two-pass adaptive steering
    # ============================================================
    if RUN_TWO_PASS:
        print("\n" + "="*70)
        print("EXPERIMENT 2: Two-pass adaptive steering")
        print("="*70)

        # Try different alpha combinations
        for av in [-1, -2, -3, 0, 1]:
            for ai in [1, 2, 3, 0, -1]:
                preds = predict_two_pass(model, tokenizer, train_items, test_items,
                                          sd_standard, av, ai, "kcast")
                r = evaluate(test_items, preds)
                results_summary.append({"exp": "two_pass", "alpha_valid": av, "alpha_invalid": ai, **r})
                if r["score"] > 18:  # only print promising ones
                    print_result(f"2pass(v={av},i={ai})", r)

    # ============================================================
    # EXPERIMENT 3: Ensemble
    # ============================================================
    if RUN_ENSEMBLE:
        print("\n" + "="*70)
        print("EXPERIMENT 3: Ensemble across alphas")
        print("="*70)

        for alpha_set in [[1, 2, 3], [2, 3, 4], [1, 3, 5], [-1, 0, 1, 2, 3],
                          [0.5, 1, 2], [-2, -1, 0, 1, 2]]:
            preds = predict_ensemble(model, tokenizer, train_items, test_items,
                                      sd_standard, alpha_set, "kcast")
            r = evaluate(test_items, preds)
            print_result(f"ens({alpha_set})", r)
            results_summary.append({"exp": "ensemble", "alphas": alpha_set, **r})

    # ============================================================
    # EXPERIMENT 4: Bias-countering ICL
    # ============================================================
    if RUN_BIAS_COUNTERING_ICL:
        print("\n" + "="*70)
        print("EXPERIMENT 4: Bias-countering ICL examples")
        print("="*70)

        print("Computing steering data (bias-countering ICL)...")
        sd_bias = compute_steering_data(model, tokenizer, train_items, target_layers, "bias_countering")

        for alpha in [0, 1, 2, 3, -1, -2, -3]:
            preds = predict_dataset(model, tokenizer, train_items, test_items,
                                     sd_bias, alpha, "kcast", icl_mode="bias_countering")
            r = evaluate(test_items, preds)
            print_result(f"biasICL α={alpha:>5.1f}", r)
            results_summary.append({"exp": "bias_icl_kcast", "alpha": alpha, **r})

        print("\n  -- All-quadrant ICL --")
        sd_allq = compute_steering_data(model, tokenizer, train_items, target_layers, "all_quadrants")
        for alpha in [0, 1, 2, 3, -1, -2]:
            preds = predict_dataset(model, tokenizer, train_items, test_items,
                                     sd_allq, alpha, "kcast", icl_mode="all_quadrants")
            r = evaluate(test_items, preds)
            print_result(f"allqICL α={alpha:>5.1f}", r)
            results_summary.append({"exp": "allq_icl_kcast", "alpha": alpha, **r})

    # ============================================================
    # FINAL SUMMARY
    # ============================================================
    print("\n" + "="*70)
    print("TOP 10 RESULTS")
    print("="*70)
    results_summary.sort(key=lambda x: x["score"], reverse=True)
    for i, r in enumerate(results_summary[:10]):
        exp = r.get("exp", "?")
        alpha_info = ""
        if "alpha" in r: alpha_info = f" α={r['alpha']}"
        if "alpha_valid" in r: alpha_info = f" αv={r['alpha_valid']} αi={r['alpha_invalid']}"
        if "alphas" in r: alpha_info = f" αs={r['alphas']}"
        print(f"  #{i+1}: {exp}{alpha_info}  ACC={r['accuracy']:.2f}  TCE={r['tce']:.2f}  "
              f"Score={r['score']:.4f}  [VP={r['vp']:.1f} VI={r['vi']:.1f} IP={r['ip']:.1f} II={r['ii']:.1f}]")

    _write_json(results_summary, os.path.join(OUT_DIR, "all_results.json"))
    print(f"\nSaved to {OUT_DIR}/all_results.json")


if __name__ == "__main__":
    main()

Model: Qwen/Qwen2.5-3B-Instruct
Layer quarter: Q3
Multi-token: True


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Train: 960, Test: 191
Layers: 36, Steering: [18, 19, 20, 21, 22, 23, 24, 25, 26]

Computing steering data (standard ICL)...
Steering pool: 945 examples (icl_mode=standard)
  100/945 (correct=69, wrong=31)
  200/945 (correct=140, wrong=60)
  300/945 (correct=204, wrong=96)
  400/945 (correct=276, wrong=124)
  500/945 (correct=347, wrong=153)
  600/945 (correct=408, wrong=192)
  700/945 (correct=484, wrong=216)
  800/945 (correct=563, wrong=237)
  900/945 (correct=643, wrong=257)
  Total: correct=676, wrong=269
  Layer 18: ||delta|| = 0.3123
  Layer 19: ||delta|| = 0.3924
  Layer 20: ||delta|| = 0.4574
  Layer 21: ||delta|| = 0.8078
  Layer 22: ||delta|| = 0.9542
  Layer 23: ||delta|| = 1.2545
  Layer 24: ||delta|| = 2.1744
  Layer 25: ||delta|| = 2.8389
  Layer 26: ||delta|| = 3.7408

EXPERIMENT 1: Standard KCAST sweep
  50/191
  100/191
  150/191
  α=  0.0: ACC=69.11  TCE=26.04  Score=16.0819  [VP=70.8 VI=39.6 IP=74.5 II=91.7]
  50/191
  100/191
  150/191
  α=  0.5: ACC=75.39  TCE=24.2